In [18]:
import os
import shutil

# 目標資料夾：把所有測試 png 平鋪搬到這裡
target_dir = "./datasets/test/images"
os.makedirs(target_dir, exist_ok=True)

def _has_test_patient_subdirs(root: str) -> bool:
    if not os.path.isdir(root):
        return False
    try:
        subdirs = [
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ]
    except FileNotFoundError:
        return False
    return any(d.lower().startswith("test_patient") for d in subdirs)

# 依常見情況嘗試找到 test_patient* 所在位置
candidate_roots = [
    "./testing_image",            # 你目前的資料夾位置
    "./datasets/test",
    "./datasets/test/testing_image",
    "./datasets/test/42_testing_image",
    "./datasets/test/42_testing_image/testing_image",
].copy()

patient_root = None
for root in candidate_roots:
    if _has_test_patient_subdirs(root):
        patient_root = root
        break

# 找不到的話再做一次遞迴搜尋（保險用）
if patient_root is None:
    for root in candidate_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            if any(d.lower().startswith("test_patient") for d in dirnames):
                patient_root = dirpath
                break
        if patient_root is not None:
            break

if patient_root is None:
    raise FileNotFoundError(
        "找不到任何 test_patient* 資料夾；請確認資料是否已解壓/路徑是否正確。"
    )

# 收集所有圖片路徑（只看直屬的 test_patient 資料夾）
all_files = []
for patient_folder in sorted(os.listdir(patient_root)):
    patient_path = os.path.join(patient_root, patient_folder)
    if os.path.isdir(patient_path) and patient_folder.lower().startswith("test_patient"):
        for fname in sorted(os.listdir(patient_path)):
            if fname.lower().endswith(".png"):
                all_files.append(os.path.join(patient_path, fname))

# 搬移到測試集 images 資料夾（可重跑：已存在就略過）
moved = 0
skipped = 0
for src in all_files:
    dst = os.path.join(target_dir, os.path.basename(src))
    if os.path.abspath(src) == os.path.abspath(dst) or os.path.exists(dst):
        skipped += 1
        continue
    shutil.move(src, dst)
    moved += 1

print(f"patient_root={patient_root}")
print(f"找到 {len(all_files)} 張 png，搬移 {moved}，略過 {skipped}（已存在或已在目標資料夾）")

patient_root=./testing_image
找到 3367 張 png，搬移 3367，略過 0（已存在或已在目標資料夾）


In [19]:
print('測試集圖片數量 : ', len(os.listdir("./datasets/test/images")))

測試集圖片數量 :  3367


In [20]:
from ultralytics import YOLO
#模型參數參考網址:https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/default.yaml
model = YOLO('./runs/detect/train/weights/best.pt')
results = model.predict(source="./datasets/test/images/",
              save=True,
              imgsz=640,
              device=0
              )


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/3367 /home/usr1202/Projects/valve_detection/datasets/test/images/test_patient0001_0001.png: 640x640 (no detections), 9.3ms
image 2/3367 /home/usr1202/Projects/valve_detection/datasets/test/images/test_patient0001_0002.png: 640x640 (no detections), 10.5ms
image 3/3367 /home/usr1202/Projects/valve_detection/datasets/test/images/test_patient0001_0003.png: 640x640 (no detections), 13.5ms
image 4/3367 /home/usr1202/Projects/valve_detection/datasets/te

In [23]:
print(len(results))

3367


In [24]:
print('預測類別 : ',results[260].boxes.cls[0].item())
print('預測信心分數 : ',results[260].boxes.conf[0].item())
print('預測框座標 : ',results[260].boxes.xyxy[0].tolist())

預測類別 :  0.0
預測信心分數 :  0.7395336031913757
預測框座標 :  [237.21775817871094, 236.7070770263672, 258.34832763671875, 252.283447265625]


In [25]:
import csv

output_file = open('./predict_csv/predict_label.csv', 'w', newline='')
writer = csv.writer(output_file)

# header
writer.writerow(["id", "image_name", "class", "confidence", "top-left x-coordinate", "top-left y-coordinate", "bottom-right x-coordinate", "bottom-right y-coordinate"])

id_counter = 0

for i in range(len(results)):
    # Get image filename (without extension)
    filename = results[i].path.split('/')[-1].split('.png')[0]
    
    # Get the number of predicted boxes
    boxes = results[i].boxes
    box_num = len(boxes.cls.tolist())
    
    # If there are predicted boxes
    if box_num > 0:
        for j in range(box_num):
            # Extract information
            label = int(boxes.cls[j].item())             # class
            conf = float(boxes.conf[j].item())           # confidence score
            x1, y1, x2, y2 = boxes.xyxy[j].tolist()      # bounding box coordinates
            
            # write row
            writer.writerow([id_counter, filename, label, f"{conf:.4f}", int(x1), int(y1), int(x2), int(y2)])
            
            # Increment after writing each row
            id_counter += 1

# Close the output file
output_file.close()